In [1]:
# Step 0: 載入套件並設定後端 (Keras + PyTorch)
# ==========================================
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import keras
from keras import layers
from sklearn.metrics import classification_report

print(keras.backend.backend())

torch


In [2]:
# Step 1: Data Loading
# ==========================================
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_test_y = pd.read_csv('gender_submission.csv')

In [3]:
# Step 2: Data Cleaning and Feature Engineering
# ==========================================
print('====Before Fix====')
print('Check for NaN:')
print(df_train.isna().any())

df_train_cleaned = df_train.copy()
df_test_cleaned = df_test.copy()

for df in [df_train_cleaned, df_test_cleaned]:
    # 2-1. Extract titles from name feature
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    
    # 2-2. Create FamilySize feature
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    
    # 2-3. Categorical feature encoding
    df['Sex'] = df['Sex'].replace(['male','female'], [0, 1])
    df['Pclass'] = df['Pclass'].replace([1, 2, 3], [1, 0, -1])
    df['Title'] = df['Title'].replace(['Mr', 'Miss', 'Mrs', 'Master', 'Rare'], [1, 2, 3, 4, 0])

# 2-4. Handle missing values using statistics from the training set (To avoid Data Leakage)
title_age_median = df_train_cleaned.groupby('Title')['Age'].median()
global_age_median = df_train_cleaned['Age'].median() 
train_embarked_mode = df_train_cleaned['Embarked'].mode()[0]
train_fare_mean = df_train_cleaned['Fare'].mean()

print('====Train Set Missing Value Statistics====')
print("Title Age Median:\n", title_age_median)
print("Global Age Median:", global_age_median)
print("Embarked Mode:", train_embarked_mode)
print("Fare Mean:", train_fare_mean)

for df in [df_train_cleaned, df_test_cleaned]:
    # Impute Age based on Title median
    for title, median in title_age_median.items():
        df.loc[(df['Age'].isnull()) & (df['Title'] == title), 'Age'] = median
    df['Age'] = df['Age'].fillna(global_age_median)
    
    # Impute and encode Embarked
    df['Embarked'] = df['Embarked'].fillna(train_embarked_mode).replace(['C','Q','S'], [-1, 0, 1])
    # Impute Fare
    df['Fare'] = df['Fare'].fillna(train_fare_mean)

# Drop unneeded text columns
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df_train_cleaned.drop(drop_cols, axis='columns', inplace=True)
df_test_cleaned.drop(drop_cols, axis='columns', inplace=True)

====Before Fix====
Check for NaN:
PassengerId    False
Survived       False
Pclass         False
Name           False
Sex            False
Age             True
SibSp          False
Parch          False
Ticket         False
Fare           False
Cabin           True
Embarked        True
dtype: bool
====Train Set Missing Value Statistics====
Title Age Median:
 Title
0    48.5
1    30.0
2    21.0
3    35.0
4     3.5
Name: Age, dtype: float64
Global Age Median: 28.0
Embarked Mode: S
Fare Mean: 32.204207968574636


In [4]:
# Step 3: Dataset Splitting (75% Train, 25% Validation)
# ==========================================
dataCT = df_train_cleaned.shape[0]
np.random.seed(42)
indexes = np.random.permutation(dataCT)
train_index = indexes[:int(dataCT*0.75)]
valid_index = indexes[int(dataCT*0.75):]

# Split into original scale datasets
train_dataO = df_train_cleaned.loc[train_index].copy()
valid_dataO = df_train_cleaned.loc[valid_index].copy()
test_dataO = df_test_cleaned.copy()

In [6]:
# Step 4: Feature Scaling
# ==========================================
scale_cols = ['Age', 'Fare', 'FamilySize']

# Compute mean and std from the training set
mean_vals = train_dataO[scale_cols].mean()
std_vals = train_dataO[scale_cols].std()

# Create standard scale feature datasets
train_data = train_dataO.copy()
valid_data = valid_dataO.copy()
test_data = test_dataO.copy()

# Apply Z-score standardization to continuous features
train_data[scale_cols] = (train_dataO[scale_cols] - mean_vals) / std_vals
valid_data[scale_cols] = (valid_dataO[scale_cols] - mean_vals) / std_vals
test_data[scale_cols] = (test_dataO[scale_cols] - mean_vals) / std_vals

print('====After Fix====')
print(train_data.head(10))
print(test_data.head(10))

# Align feature lists
feature_cols = [col for col in train_data.columns if col != 'Survived']

# Convert to numpy arrays for neural network inputs
x_train = np.array(train_data[feature_cols]).astype(np.float32)
y_train = np.array(train_data['Survived']).astype(np.int64)

x_valid = np.array(valid_data[feature_cols]).astype(np.float32)
y_valid = np.array(valid_data['Survived']).astype(np.int64)

x_test = np.array(test_data[feature_cols]).astype(np.float32)
y_test = np.array(df_test_y['Survived']).astype(np.int64)

print('====Final Array Check====')
print("x_train has NaN:", np.isnan(x_train).any())
print("x_valid has NaN:", np.isnan(x_valid).any())
print("x_test has NaN:", np.isnan(x_test).any())
print("x_train has Inf:", np.isinf(x_train).any())
print("Feature column order:", feature_cols)
print("Dataset shapes (x_train, y_train):", x_train.shape, y_train.shape)

====After Fix====
     Survived  Pclass Sex       Age  SibSp  Parch      Fare Embarked Title  \
709         1      -1   0 -1.917715      1      1 -0.345898       -1     4   
439         0       0   0  0.114880      0      0 -0.440590        1     1   
840         0      -1   0 -0.698158      0      0 -0.491968        1     1   
720         1       0   1 -1.732934      0      1  0.008346        1     2   
39          1      -1   1 -1.141633      1      0 -0.425791       -1     2   
290         1       1   1 -0.254683      0      0  0.923176        1     2   
300         1      -1   1 -0.624246      0      0 -0.495460        0     2   
333         0      -1   0 -0.993808      2      0 -0.290945        1     1   
208         1      -1   1 -0.993808      0      0 -0.495460        0     2   
136         1       1   1 -0.772071      0      2 -0.125671        1     2   

     FamilySize  
709    0.716025  
439   -0.596196  
840   -0.596196  
720    0.059914  
39     0.059914  
290   -0.596196